# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset: [A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access the metadata as a single object
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Dataset description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant schema's `@id` fields to reference entities. Let's list the record sets and their fields.

In [ ]:
# Identify all record sets using their '@id' fields

# In `metadata`, the record sets live under the 'recordSet' attribute
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print('No record sets found in metadata.')
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"  Record set @id: {rs['@id']} | Name: {rs.get('name', 'Unnamed')}")
        # Print available fields
        fields = rs.get('field', [])
        for field in fields:
            print(f"    Field @id: {field['@id']} | Name: {field.get('name', 'Unnamed')} | DataType: {field.get('dataType', 'Unknown')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this FAIR² mental health dataset, we'll load the main survey record set. Replace `<record_set_id>` and field IDs below with those discovered above for your analysis.

In [ ]:
# Extract data from each record set into a pandas DataFrame
# Replace the following with the discovered record set '@id's

record_sets_ids = []
# Discover record set IDs
for rs in getattr(metadata, 'recordSet', []):
    record_sets_ids.append(rs['@id'])

dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in '{record_set_id}': {df.columns.tolist()}")
    print(df.head(2))

# Use the first record set as example for further processing
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    print(f"Main analysis will use record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We reference fields by their `@id`.

We'll select fields such as GAD-7 score, PHQ-9 score, or Age (if available) for numeric analysis.

In [ ]:
# EDA: Filter, normalize, group
import numpy as np

# Example field '@id's: Replace with real ids based on your overview
# Suppose 'age' field has @id 'https://api.app.sen.science/frontiers/7160411/field_age',
# 'gad_7_score' has @id 'https://api.app.sen.science/frontiers/7160411/field_gad7',
# 'level_of_education' has @id 'https://api.app.sen.science/frontiers/7160411/field_education'

# To simulate, try guessing column names from loaded DataFrame
df = dataframes[main_record_set_id]

numeric_fields = [col for col in df.columns if ('score' in col.lower() or 'age' in col.lower())]
print(f"Numeric fields in dataset: {numeric_fields}")

# Let's choose one, e.g., 'GAD_7_score' or closest
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    numeric_field = None

threshold = 10  # Example threshold for filtering
if numeric_field is not None:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (by @id):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by an attribute, such as level_of_education, if available
    # Find a categorical/group field
    group_fields = [col for col in df.columns if 'education' in col.lower() or 'gender' in col.lower()]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_field)
        print(f"Grouped (mean) {numeric_field} by {group_field}:")
        print(grouped_df)
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the chosen numeric field, and optionally a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by group_field
    if group_fields:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey dataset provides rich demographic and psychological information, referencing all entities via their Croissant `@id`s.
- Key numeric fields such as GAD-7 and PHQ-9 scores support quantitative analysis and filtering.
- Visualizations highlight the distribution of mental health indicators and relationships with demographic groupings.
- All data access and processing via `mlcroissant` is based on Croissant schema entity `@id` references for reproducibility and clarity.